# India House Price Prediction — ML Pipeline
EDA -> Preprocessing -> Model Training -> Evaluation -> Export

Dataset: synthetic, generated by `generate_dataset.py` (see that file's docstring —
this sandbox has no internet access to pull a real Kaggle/MagicBricks dataset, so
the data is generated from the same city/locality rate table already live in the
frontend, plus realistic noise, missing values, and outliers). Swap in a real
dataset with matching columns (`data_dictionary.md`) to replace this later.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("data/india_housing_raw.csv")
df.shape

## 1. EDA

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

In [ ]:
df.groupby("city")["price_per_sqft"].median().sort_values(ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
df.boxplot(column="price_per_sqft", by="city", ax=ax, rot=45)
plt.suptitle("")
plt.title("Price per sq.ft by city (raw, pre-cleaning)")
plt.tight_layout()
plt.show()

In [ ]:
df["price"].plot(kind="hist", bins=80, figsize=(8,4), title="Price distribution (raw, includes injected outliers)")
plt.show()

## 2. Preprocessing (clean, impute, remove outliers, encode, scale)

See `preprocessing.py` for the reusable pipeline. Summary of what it does:
- drops duplicates
- imputes missing bathrooms/total_floors/property_age_years with the per-city median
- removes price/sq.ft outliers using a per-city IQR rule (a global IQR would unfairly flag every Mumbai listing)
- one-hot encodes city/locality/furnishing/property type/age band
- standard-scales numeric features

In [ ]:
from preprocessing import load_and_prepare

data = load_and_prepare()
print("Train:", data["X_train"].shape, " Test:", data["X_test"].shape)
print("Features after encoding:", len(data["feature_names"]))

In [ ]:
clean_df = data["df_clean"]
clean_df["price"].plot(kind="hist", bins=80, figsize=(8,4), title="Price distribution (cleaned)")
plt.show()

## 3. Train & compare models

Linear Regression (baseline), Random Forest, XGBoost (falls back to sklearn's GradientBoostingRegressor if xgboost isn't installed in this environment), and a Neural Network (MLP).

In [ ]:
%run train.py

## 4. Evaluation summary

In [ ]:
import json
metrics = json.load(open("models/metrics.json"))
results_df = pd.DataFrame(metrics["results"]).T[["rmse","mae","r2","train_seconds"]]
results_df.sort_values("rmse")

In [ ]:
results_df.sort_values("rmse")["r2"].plot(kind="bar", figsize=(7,4), title="R2 by model")
plt.ylabel("R2")
plt.tight_layout()
plt.show()

Best model: see `models/best_model.txt` - chosen by lowest RMSE on the held-out test set.

## 5. Feature importance - "what is driving this price"

In [ ]:
fi = json.load(open("models/feature_importance.json"))
print("Source model:", fi["source_model"])
fi_df = pd.DataFrame(fi["importances"][:15])
fi_df.plot(kind="barh", x="feature", y="importance", figsize=(8,6), legend=False)
plt.gca().invert_yaxis()
plt.title("Top 15 feature importances")
plt.tight_layout()
plt.show()

## 6. Sample prediction with confidence range

This is exactly what `POST /predict` will call (via `predict.py`'s `PricePredictor`).

In [ ]:
from predict import PricePredictor

predictor = PricePredictor()
example = {
    "city": "bengaluru", "locality": "Whitefield", "area_sqft": 1250,
    "bhk": 3, "bathrooms": 2, "floor": 6, "total_floors": 14,
    "property_age_years": 3, "age_band": "1-5",
    "furnishing_status": "semi", "property_type": "apartment",
    "parking": 1, "lift": 1, "security": 1, "power_backup": 1,
    "gym": 0, "pool": 0, "clubhouse": 0, "garden": 0,
}
predictor.predict(example)

## 7. Export

All trained models, the fitted preprocessor, metrics, and feature importances are already saved to `models/` by `train.py` (joblib `.pkl` + `.json`). The FastAPI backend loads `models/preprocessor.pkl` + `models/<best_model>.pkl` at startup for live inference - no retraining needed to serve predictions.

To retrain as new data comes in: replace/append to `data/india_housing_raw.csv` and re-run `python train.py`.